# Greedy Choices and Exchange Reasoning


```{contents}
:local:
:depth: 2
```


```{index} Identify the Objective That Makes the Exchange Work
```



```{index} Identify the Objective That Makes the Exchange Work
```

## Identify the Objective That Makes the Exchange Work



Meeting scheduling can ask for the largest number of compatible meetings or for the highest total meeting value. Those are different optimization problems. Earliest finishing time supports the first objective under the half-open interval model; it does not automatically maximize a sum of business values.

Suppose meetings are A=[0,2), B=[2,4), and C=[0,4). Choosing A then B yields two meetings, whereas choosing C yields one. Touching endpoints are compatible because the first interval excludes its finishing point. If the business instead requires a cleanup gap, feasibility changes and must be reflected in both selection and tests.

The exchange proof begins with an optimal count schedule and compares its first meeting with the earliest-finishing eligible one. Replacing that first meeting by one that finishes no later leaves at least as much room for the remaining meetings. The count is preserved, so some optimum begins with the greedy choice. Repeat the argument for the remaining compatible problem. The proof depends on the objective and feasibility relation, not on the persuasive phrase make the best local choice.

| Changed requirement | What must be reconsidered? |
| --- | --- |
| Maximize number of meetings | The count exchange argument applies under the stated interval assumptions |
| Maximize meeting value | An early low-value meeting may block a more valuable one |
| Require cleanup time | Compatibility must include the gap |
| Allow zero-duration entries | Define their eligibility and ordering explicitly |

Guided counterexample: give C value 100 and A and B value one each. The count-optimal schedule has value two, while C alone has value 100. Explain why this disproves reuse of the count rule for the value objective without contradicting its original correctness proof. Chapter 25 develops optimization states when a local exchange rule is not sufficient.

For related explanation, see the further reading below.[^development-reading]


```{index} Greedy Choice and Correctness
```

<span id="greedy-choice-and-correctness"></span>


A greedy algorithm chooses locally and commits. Specify the objective, feasible choices, tie policy, and reason a choice can belong to an optimum. Feasibility alone is insufficient.

```{index} Schedule the Maximum Number of Meetings
```

## Schedule the Maximum Number of Meetings




```{index} Schedule the Maximum Number of Meetings
```

One room holds fixed, positive-duration meetings [start,end). Back-to-back meetings are compatible. Maximize count; every meeting has equal importance for this objective. Select compatible meetings in nondecreasing finish order. Validate unique IDs and Start < End; empty input returns an empty selection.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine(string.Join(",",chosen.Select(m=>m.Id)));
Console.WriteLine($"count={chosen.Length}");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


A,D,E
count=3


The selected meetings never overlap, and lastEnd is the end of the last selected meeting. Sorting costs Θ(n log n); the selection scan is Θ(n). This implementation returns up to n selected records and uses sorting storage, so it does not claim constant total auxiliary space. Equal finish times use deterministic start/ID tie breaks; another compatible tie rule can also achieve the optimal count.

**Common mistake.** Sorting by start time or by duration instead of by finish time. Earliest start can pick the long meeting `C` and block three others; earliest finish is the one rule the exchange argument supports for the count objective.


```{index} Trace Accepted and Rejected Choices
```

## Trace Accepted and Rejected Choices




```{index} Trace Accepted and Rejected Choices
```

A useful trace includes rejected choices and the evolving feasibility boundary.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
int lastEnd=int.MinValue;
foreach(var m in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
{
 bool accept=m.Start>=lastEnd;
 Console.WriteLine($"{m.Id} [{m.Start},{m.End}): {(accept?"accept":"reject")}");
 if(accept)lastEnd=m.End;
}
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


A [1,4): accept
B [3,5): reject
C [0,6): reject
D [5,7): accept
E [8,9): accept


A feasible schedule can still have too few meetings. Choosing the earliest starting long meeting, for example, can block several shorter meetings. Always distinguish a feasibility invariant from an optimality argument.

**Common mistake.** Accepting a meeting whose start equals `lastEnd` as a conflict, or rejecting it. With half-open intervals a meeting starting exactly when the last one ends is compatible, which is why the test is `m.Start >= lastEnd` and not `>`.


```{index} Exchange the First Choice
```

## Exchange the First Choice




```{index} Exchange the First Choice
```

Let g be the globally earliest-finishing available meeting. Take an optimal schedule with first meeting o. Since end(g)≤end(o), replace o with g. Every later meeting starts at or after end(o), hence also at or after end(g); feasibility and count are unchanged. Thus some optimal schedule begins with g. Apply the same argument to the remaining compatible meetings.

The positive-duration assumption ensures g cannot also be a later meeting in the original schedule. After the exchange, the residual problem has the same shape. Induction on the number of available meetings establishes optimal count, rather than merely saying that g leaves room. The loop invariant can be strengthened: the accepted prefix can be extended to an optimal solution for the original instance.

```{index} A Verified Counterexample for a Different Objective
```

## A Verified Counterexample for a Different Objective




```{index} A Verified Counterexample for a Different Objective
```

Change the goal from maximum count to maximum total meeting value. Earliest finish now fails: two short meetings can be worth less than one long meeting. Compute both feasible solutions rather than asserting a failure from an incomplete schedule.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",0,2,4),new MeetingSchedule.Meeting("B",2,4,4),new MeetingSchedule.Meeting("C",0,4,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine($"earliest finish: {string.Join(",",chosen.Select(m=>m.Id))}, count={chosen.Length}, value={chosen.Sum(m=>m.Value)}");
Console.WriteLine("alternative C: count=1, value=9; both schedules are feasible");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


earliest finish: A,B, count=2, value=8
alternative C: count=1, value=9; both schedules are feasible


Count is maximized by A,B, but their value eight loses to C’s value nine. This does not refute the count algorithm; it refutes reusing it for weighted value. Chapter 25 develops dynamic programming for problems where a local commitment lacks this exchange property.

**Common mistake.** Keeping the earliest-finish rule and adding the values afterward. Value eight loses to nine here, and no tie-break repairs it; a changed objective needs a changed argument, which Chapter 25 supplies.


```{index} Another Greedy Trap: Coin Counts
```

## Another Greedy Trap: Coin Counts




```{index} Another Greedy Trap: Coin Counts
```

For denominations 1,3,4 and amount six, choosing the largest affordable coin yields 4+1+1. The feasible alternative 3+3 uses fewer coins. Availability of coin one guarantees completion, not optimality.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int remaining=6;var coins=new List<int>();
foreach(int denomination in new[]{4,3,1})
 while(remaining>=denomination){coins.Add(denomination);remaining-=denomination;}
Console.WriteLine($"greedy={string.Join("+",coins)}, count={coins.Count}");
Console.WriteLine("alternative=3+3, count=2");

greedy=4+1+1, count=3
alternative=3+3, count=2


A greedy rule needs a proof tied to the denomination system and objective. A counterexample invalidates a universal claim; successful examples do not prove one. Test small complete alternatives, equal keys, empty inputs, and changed objectives before trusting a rule.

```{index} Exercise
```

## Exercise




```{index} Exercise
```

Choose the maximum-count compatible schedule, including a negative start time and a back-to-back boundary. Explain the exchange argument and reject a zero-duration interval.

This starter includes its supporting types and operations below the marked client area. Run it from a fresh session without running a previous demonstration or revealing the answer. Your task is to write the requested calls, assertions and interpretation, not to recreate the implementation. Compare the ordinary result and each boundary result against the contract before inspecting the hidden solution.


In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",-3,0),new MeetingSchedule.Meeting("B",0,2),new MeetingSchedule.Meeting("C",-2,3)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.
// Cell-local support: inspect the operations, then write the client checks above.
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
var meetings=new[]{new MeetingSchedule.Meeting("A",-3,0),new MeetingSchedule.Meeting("B",0,2),new MeetingSchedule.Meeting("C",-2,3)};
Console.WriteLine(string.Join(",",MeetingSchedule.Select(meetings).Select(m=>m.Id)));
try{MeetingSchedule.Select(new[]{new MeetingSchedule.Meeting("Bad",2,2)});}catch(ArgumentException){Console.WriteLine("zero duration rejected");}
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


A,B
zero duration rejected


The output is `A,B` and `zero duration rejected`. A ends at 0 and B starts at 0, so under half-open intervals they are compatible; C overlaps both and is rejected. The zero-duration meeting fails the `Start < End` contract before any scheduling happens.


```{index} Greedy Optimization Lab
```

<span id="greedy-optimization-lab"></span>


Use one facilities theme to connect scheduling, cable planning, and delivery routing. Their objectives differ; keep graph direction, weight units, and feasibility policies explicit. Each example includes its own setup.

Download the complete implementations and independent checks: [GreedyChecks.cs](../../demos/24/GreedyChecks.cs).


```{index} Validate the Meeting Objective
```

## Validate the Meeting Objective




```{index} Validate the Meeting Objective
```

Compare earliest-finish count with complete subsets on a tiny instance. Exhaustive verification is a small-input test oracle, not the scalable algorithm.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
int best=0;
for(int mask=0;mask<(1<<meetings.Length);mask++)
{
 var subset=meetings.Where((m,i)=>(mask&(1<<i))!=0).OrderBy(m=>m.Start).ToArray();
 bool feasible=true;
 for(int i=1;i<subset.Length;i++)if(subset[i].Start<subset[i-1].End)feasible=false;
 if(feasible)best=Math.Max(best,subset.Length);
}
int greedy=MeetingSchedule.Select(meetings).Length;
if(greedy!=best)throw new Exception("Count mismatch.");
Console.WriteLine($"greedy count={greedy}, exhaustive count={best}");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


greedy count=3, exhaustive count=3


Exhaustive enumeration is exponential, so keep it tiny. The exchange proof explains why the efficient algorithm works for all valid sizes. Changing to weighted values breaks that proof; retain the counterexample from Section 22.1.

**Common mistake.** Trusting the exhaustive check on a large instance. With twenty meetings the subset loop runs about a million times, and with forty it cannot finish; the oracle is for tiny instances, and the proof covers the rest.


```{rubric} Footnotes
```


[^2401_greedy_choice-1]: Earliest-finish interval scheduling maximizes count under the fixed one-room, positive-duration model; weighted objectives require different reasoning.
[^2401_greedy_choice-2]: Greedy correctness requires more than feasibility. An exchange argument connects accepted choices to an optimal complete solution.

[^development-reading]: Further reading: [Sedgewick and Wayne, Minimum Spanning Trees](https://algs4.cs.princeton.edu/43mst/index.php). The worked business scenarios and guided reasoning here are original teaching examples.
